# RoBERTa runs for the LIAR project (code + viva notes)

Har code cell ke **upar** ek VIVA Q&A cell hai, aur code ke andar har command ke upar uska comment. Assessment me inhi se jawab likh sakte ho.

**VIVA Q: Is notebook ka maqsad kya hai?**
**A:** Project ka sawal tha: "kya chhota fine-tuned model (110M-125M parameters) ek bade prompted LLM (7.6B) se behtar hai, jab accuracy aur cost dono gino?" RoBERTa BERT ka behtar-trained version hai; usko dono tarah (sirf statement, aur statement + speaker info) train karke baaki 5 models se compare karna tha. Ye notebook wahi 2 runs Colab ke T4 GPU par chalata hai.

**VIVA Q: Colab / T4 GPU kyun, laptop par kyun nahi?**
**A:** Laptop me GPU nahi hai. RoBERTa (124,650,246 parameters) T4 GPU par har run ~3.4-3.9 minute leta hai; CPU par ye ghanto lagte. Colab free hai aur internet default on rehta hai.

**VIVA Q: RoBERTa BERT se alag kaise hai?**
**A:** Architecture lagbhag same (transformer encoder) par training behtar: ~160 GB text (BERT ~16 GB), dynamic masking, next-sentence-prediction hata diya, bade batches aur zyada steps, aur byte-level BPE vocabulary (~50k tokens, BERT ka WordPiece ~30k). roberta-base cased hai, bert-base-uncased lowercase karta hai. Parameters: 124.65M vs 109.49M.

### VIVA Q&A: Cell 1 (clone aur install)

**Q: git clone kyun, aur kaun si branch?**
**A:** Colab ek khaali computer hai; hamara code usme nahi hota. Clone se repo ki main branch aati hai, jisme `finetune/train_bert.py` (BERT/RoBERTa ki shared training script), `common/` (labels, scoring) aur `liar_data/` (LIAR ki TSV files) hain.

**Q: `%cd repo` ka kya matlab?**
**A:** Folder badalna. `%` IPython ka magic command hai (shell ka `cd` har cell me bhool jaata), isliye `%cd` use karte hain. Iske baad `python -m finetune.train_bert` sahi jagah se chalta hai.

**Q: pip install me `datasets<4` kyun?**
**A:** LIAR ka Hugging Face loading script naye `datasets` (4+) me band hai, toh loader fail hota hai. Script phir repo ke `liar_data/` ki TSV files par fallback karti hai (output me dikhta hai: "falling back to the official LIAR zip", "Reading LIAR from liar_data/").

**Q: Cell ke end me laal "dependency resolver" error kya hai?**
**A:** Pip ka warning hai (`gcsfs requires fsspec==2025.12.0`, jabki 2025.3.0 installed hai). Colab ke apne pre-installed package se takkar hai; humare kaam par koi asar nahi hota, training normal chalti hai.

**Q: transformers, scikit-learn, accelerate kis kaam ke hain?**
**A:** transformers = pre-trained BERT/RoBERTa aur tokenizer; scikit-learn = accuracy, macro-F1, confusion matrix; accelerate = Hugging Face ka training helper (mixed precision wagairah).

In [1]:
# Step 1: GitHub se project ka code Colab ke andar download (clone) karo. 'repo' naam ka folder banega.
!git clone https://github.com/arunchowdary-1/Fake-News-Detection-Fine-tuned-Transformers-vs-Zero-shot-LLMs.git repo
# Step 2: us 'repo' folder ke andar jao. Aage ki saari commands wahin se chalti hain (python -m finetune... ke liye zaroori).
%cd repo
# Step 3: libraries install. -q = kam output, -U = latest version, transformers = BERT/RoBERTa, scikit-learn = metrics,
# accelerate = Hugging Face ka training helper, datasets<4 = purana datasets (naye version me LIAR ka loading script band hai).
!pip install -q -U transformers scikit-learn accelerate "datasets<4"

Cloning into 'repo'...
remote: Enumerating objects: 223, done.
remote: Counting objects: 100% (223/223), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 223 (delta 96), reused 182 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (223/223), 1.12 MiB | 11.57 MiB/s, done.
Resolving deltas: 100% (96/96), done.
/content/repo
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


### VIVA Q&A: Cell 2 (smoke test)

**Q: Smoke test kyun?**
**A:** Asli run ~4 minute leta hai. Pehle 200 rows par 1 epoch chalake dekhte hain ki data load, model load, training, evaluation aur result files banana, sab bina error chalta hai. Galti ho toh 1 minute me pata chal jaata hai.

**Q: Smoke test ka score (accuracy 0.163, macro-F1 0.047) itna kam kyun?**
**A:** Model ne sirf 200 rows dekhe, wo bhi ek baar; isliye lagbhag chance (6 classes me random = 16.7%) hai. Ye expected hai; maqsad score nahi, pipeline check karna tha.

**Q: Output ki pehli line `Device: Tesla T4 | fp16: True` kya batati hai?**
**A:** GPU mil gayi (T4) aur fp16 (half precision) on hai, jo T4 par training ~2x tez aur kam memory wali banata hai. Agar yahan `Device: CPU` aaye toh runtime GPU par set nahi hai; tab rok ke Runtime > Change runtime type > T4 GPU chuno.

**Q: Output me `UNEXPECTED lm_head...` aur `MISSING classifier...` kya hai, kya galti hai?**
**A:** Galti nahi. Pre-trained RoBERTa me language-modelling head (`lm_head`) hota hai jo classification me kaam nahi aata, isliye "UNEXPECTED" (ignore). Naya 6-class `classifier` head hum jodte hain, jo pre-trained file me nahi tha, isliye "MISSING" (random shuru hota hai aur fine-tuning me seekhta hai).

**Q: `HF_TOKEN` warning aur `GradScaler` FutureWarning kya hain?**
**A:** HF_TOKEN: bina login ke download me thodi speed limit; public model hai, toh koi dikkat nahi. GradScaler: `torch.cuda.amp` ka naam purana ho gaya, naya naam alag hai; sirf chetavni hai, kaam normal chalta hai.

**Q: Parameters 124,650,246 kahan se?**
**A:** roberta-base ke ~124.6M weights (embedding + 12 transformer layers) plus naya classification head. BERT-base 109,486,854 hai.

**Q: `--run-name roberta_smoke` alag kyun?**
**A:** Taaki smoke test ki files asli `roberta_text` files ko overwrite na karein, aur zip me bhi na jayein.

In [2]:
# SMOKE TEST: sirf 200 rows par 1 epoch, ye dekhne ke liye ki poori pipeline (data, model, training, results files) bina error chalti hai.
# -m finetune.train_bert = repo ki training script chalao | --mode text = sirf statement (metadata nahi)
# --model-name roberta-base = Hugging Face ka pre-trained RoBERTa | --run-name roberta_smoke = results ka alag naam (asli files na bigdein)
# --epochs 1 = bas ek baar poora data | --max-train 200 = sirf 200 training rows
!python -m finetune.train_bert --mode text --model-name roberta-base --run-name roberta_smoke --epochs 1 --max-train 200

Device: Tesla T4 | fp16: True | mode: text | model: roberta-base
Loading LIAR dataset from Hugging Face...
README.md: 100% 5.16k/5.16k [00:00<00:00, 9.89MB/s]
liar.py: 100% 6.41k/6.41k [00:00<00:00, 12.2MB/s]
README.md: 100% 5.16k/5.16k [00:00<00:00, 17.4MB/s]
liar.py: 100% 6.41k/6.41k [00:00<00:00, 17.1MB/s]
load_liar() failed (Failed to load LIAR dataset: https://www.cs.ucsb.edu/~william/data/liar_dataset.zip); falling back to the official LIAR zip
Reading LIAR from liar_data/
Train 10269 | val 1284 | test 1283
config.json: 100% 481/481 [00:00<00:00, 2.16MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 119kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 1.36MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 1.07MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 1.68MB/s]

model.safetensors: downloading bytes:  30% 149M/499M [00:01<00:02, 147MB/s, 11.2MB/s  ]
model.safetensors: downloading bytes:  39% 192M/499M [00:01<00:01, 169MB/s, 16.6MB/s  ]
model.safetensors: downloading byt

### VIVA Q&A: Cell 3 (asli runs: roberta_text aur roberta_meta)

**Q: Dono runs me kya fark hai?**
**A:** Sirf input ka. `text`: sirf statement (max 128 tokens). `meta`: `speaker: ... | party: ... | subject: ... | history: a,b,c,d,e </s> statement` (max 256 tokens, kyunki prefix lamba hai). Baaki sab same, isliye fark seedha metadata ka asar dikhata hai (ablation).

**Q: RoBERTa me `[SEP]` ki jagah `</s>` kyun dikhta hai?**
**A:** Har model ka apna separator token hota hai: BERT = `[SEP]`, RoBERTa = `</s>`. Script `tokenizer.sep_token` se sahi wala lagati hai.

**Q: Training setup kya hai?**
**A:** AdamW, learning rate 2e-5, weight decay 0.01, batch 16, 3 epochs, 10% warm-up fir linear decay, gradient clipping 1.0, fp16, aur inverse-frequency class weights. Best epoch validation macro-F1 se chuna gaya (dono runs me epoch 3, yaani training aur chalti toh thoda sudhar sambhav tha).

**Q: Class weights, early stopping aur seed 42 kyun?**
**A:** Class weights: LIAR unbalanced hai, aur macro-F1 har class barabar tolta hai. Early stopping (patience 2): validation sudhar ruke toh overfit se pehle ruko. Seed 42: result dobara aaye; sab models me same, fair comparison ke liye (GPU ki wajah se bahut chhota fark aa sakta hai).

**Q: Hyperparameters BERT wale hi kyun?**
**A:** Taaki BERT vs RoBERTa ka fark model ka ho, settings ka nahi. Teeno (BERT, RoBERTa, LSTM) shared `train_bert.py` / shared scoring code se score hote hain.

**Q: Results kya aaye?**
**A:** `roberta_text`: 300-row subset par accuracy 0.283, macro-F1 0.282; full test (1,283 rows) par 0.288 / 0.291; train 3.4 min; 14.4 ms per statement; GPU memory 2,429 MB (train) / 2,131 MB (inference). `roberta_meta`: subset 0.473 / 0.483; full test 0.461 / 0.467; train ~3.9 min; 17.3 ms.

**Q: Metadata se accuracy 28% se 47% kyun badhi, aur kya ye bharosemand hai?**
**A:** Speaker, party aur subject asli signal dete hain. Par credit-history numbers me statement ka apna label shamil hota hai, toh model jawab ka hissa history se padh leta hai (leakage). Isliye 47% ko "speaker jaanne ki value" kaho, behtar language understanding nahi. LLM se fair comparison text-only run hai.

**Q: Text-only RoBERTa BERT se behtar hai?**
**A:** Nahi, statistically tied hai (28.3% vs 29.3% subset; McNemar p = 0.80). 300 rows par har accuracy ~+/-5 points uncertain hoti hai.

**Q: Ek hi seed ka kya nuksan?**
**A:** Run-to-run variation naapa nahi gaya; report ki limitations me bhi likha hai.

In [3]:
# RUN 1: sirf statement. Isse banti hain results/roberta_text.json aur results/roberta_text_preds.csv. Naam bilkul yehi rakhna (cost table isi naam se padhta hai).
!python -m finetune.train_bert --mode text --model-name roberta-base --run-name roberta_text
# RUN 2: metadata + statement (speaker, party, subject, credit history). Isse banti hain results/roberta_meta.json aur roberta_meta_preds.csv.
# --mode meta me input lamba hota hai, isliye script max length 128 ki jagah 256 tokens leti hai.
!python -m finetune.train_bert --mode meta --model-name roberta-base --run-name roberta_meta

Device: Tesla T4 | fp16: True | mode: text | model: roberta-base
Loading LIAR dataset from Hugging Face...
load_liar() failed (Failed to load LIAR dataset: https://www.cs.ucsb.edu/~william/data/liar_dataset.zip); falling back to the official LIAR zip
Reading LIAR from liar_data/
Train 10269 | val 1284 | test 1283
Loading weights: 100% 197/197 [00:00<00:00, 3786.50it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

### VIVA Q&A: Cell 4 (zip aur download)

**Q: Sirf 4 files kyun?**
**A:** Hamare shared contract ke hisaab se har model ke 2 files hote hain: `results/<naam>.json` (accuracy, macro-F1, confusion matrix, params, time, latency, cost) aur `results/<naam>_preds.csv` (id, y_true, y_pred, latency_ms). Do runs x 2 = 4. Cost table aur error analysis inhi se banti hain.

**Q: `zip -j` ka `-j` kya karta hai?**
**A:** "junk paths": zip me `results/` folder nahi banta, sirf file ke naam aate hain. Isse zip kholne par files seedha mil jaati hain.

**Q: `files.download` kya karta hai?**
**A:** Colab ke server se file tumhare computer ke Downloads me bhejta hai (Colab ka runtime band hone par uski saari files mit jaati hain, isliye download zaroori hai).

**Q: Iske baad kya?**
**A:** Files repo ke `results/` me jaati hain (PR), phir `python -m analysis.error_analysis --run roberta_text` se error analysis, aur `python -m llm.cost_analysis` se cost table me RoBERTa ki row aati hai.

In [4]:
# Chaar result files ki ek zip banao. -j = folder ke path hata ke sirf file ke naam rakho (zip me results/ folder nahi banega).
!zip -j roberta_results.zip results/roberta_text.json results/roberta_text_preds.csv results/roberta_meta.json results/roberta_meta_preds.csv
# google.colab ka 'files' helper: Colab ke andar se file tumhare computer par download karne ke liye.
from google.colab import files
# Browser me zip download shuru karta hai (popup/Downloads bar dekho; block ho toh Allow karo).
files.download("roberta_results.zip")

  adding: roberta_text.json (deflated 68%)
  adding: roberta_text_preds.csv (deflated 62%)
  adding: roberta_meta.json (deflated 68%)
  adding: roberta_meta_preds.csv (deflated 60%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>